# Google Gemini (google-genai) – Hands-on Tutorial

This notebook explains the core Google Gemini (`google-genai`) API functions
used to build a chat application.

By the end, you will understand:
- How to connect to Gemini
- How prompts are structured
- How responses are generated
- How to control output format
- How to build chat-style interactions


## 1. Installation

Install the required libraries:

```bash
pip install google-genai python-dotenv
```

## 2. API Key Setup

Create a `.env` file in the same folder:

```
GEMINI_API_KEY=your_api_key_here
GEMINI_MODEL=gemini-3.1-flash-lite
```

We will load this securely using `python-dotenv`.

In [ ]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")

## 3. genai.Client()

The `Client` object authenticates your requests and allows you to interact
with Gemini models.

In [ ]:
client = genai.Client(api_key=API_KEY)

## 3b. Which models can I use?

Model names change every few months (last year we used `gemini-2.5-flash-lite`). Never guess - ask the API. Pick a *flash* or *flash-lite* model: they are fast and have a free tier.

In [ ]:
for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

## 4. Single-turn Prompt

Use `generate_content()` to send a prompt and receive a response.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain artificial intelligence in one paragraph."
)

print(response.text)

## 5. Structured Messages with types.Content()

Gemini expects structured messages with roles and parts.

In [ ]:
message = types.Content(
    role="user",
    parts=[types.Part.from_text(text = "What is quantum computing?")]
)

response = client.models.generate_content(
    model=MODEL,
    contents=[message]
)

print(response.text)

## 6. types.Part.from_text()

Each message can contain multiple parts. Here we use text parts.

In [ ]:
part = types.Part.from_text(text = "Explain neural networks simply.")
print(part.text)

## 7. Multi-turn Conversation

Passing multiple `Content` objects simulates a conversation.

In [ ]:
conversation = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text = "What is machine learning?")]
    ),
    types.Content(
        role="user",
        parts=[types.Part.from_text(text = "Explain it like I'm 10.")]
    )
]

response = client.models.generate_content(
    model=MODEL,
    contents=conversation
)

print(response.text)

## 8. Output Control with GenerateContentConfig

This allows forcing JSON or other output formats.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Give a quiz question as JSON.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

print(response.text)

## 8b. System instructions

A *system instruction* sets the bot's personality for the whole chat. This is how we will build the Socratic mode in Week 2 - much more reliable than pasting rules into the user message.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is a Q-table?",
    config=types.GenerateContentConfig(
        system_instruction="You are a tutor. Answer in at most 2 sentences, then ask the student one question.",
        temperature=0.4,
    ),
)
print(response.text)

## 9. Error Handling

Always wrap model calls with try-except.

In [ ]:
try:
    response = client.models.generate_content(
        model="invalid-model",
        contents="Hello"
    )
    print(response.text)
except Exception as e:
    print("Error:", e)

## Summary

You learned:
- How to initialize Gemini
- How to send prompts
- How messages are structured
- How to control output format
- How to handle errors

This forms the foundation for building chatbots, agents, and AI tutors.